# Candidate recovery metric: daily step count

Steps are the most direct wearable proxy for "back to normal function", and the only candidate
here that can be measured both before and after surgery on the same instrument. This notebook
does not model recovery -- it asks whether the data exist densely enough, on enough of the
cohort, at the right times, for a steps-based outcome to be definable at all.

The decision this feeds is which single outcome the project commits to. The number that matters
most is the paired count: patients with enough pre-operative days to establish a baseline **and**
enough post-operative days to measure a return to it.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_steps")
CDR, CDR_INDEX = cfg.CDR, cfg.CDR_INDEX
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = f"""
    activity AS (
        SELECT
            a.person_id,
            DATE_DIFF(a.date, i.index_date, DAY) AS rel_day,
            a.steps
        FROM `{CDR}`.activity_summary AS a
        JOIN index_surgery AS i ON i.person_id = a.person_id
        WHERE a.steps IS NOT NULL
    )"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# 2-coverage

### Does the Fitbit activity table have the columns this notebook assumes?

Table and column names are the one thing a notebook cannot infer, and a wrong guess surfaces as
an opaque BigQuery error a long way into a run. This probe reads only `INFORMATION_SCHEMA`, which
is schema metadata rather than participant data.

In [ ]:
activity_columns = cached_gbq(
    srp_common.columns_sql(cfg, ["activity_summary"]), "activity_summary_columns"
)

if activity_columns.empty:
    print("activity_summary not found in the CDR -- check the table name in the "
          "All of Us data dictionary for this release before going further.")
else:
    print(activity_columns.to_string(index=False))

### How many cohort patients have any daily step record at all?

This is the ceiling on every steps-based outcome: a patient with no activity row can never
contribute one, whatever window is chosen. The gap between the cohort count and this count is
pure, irrecoverable loss.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(*) FROM index_surgery)                AS patients_with_index_date,
        (SELECT COUNT(DISTINCT person_id) FROM activity)    AS patients_with_any_steps,
        (SELECT COUNT(*) FROM activity)                     AS step_days
"""

cov = cached_gbq(coverage_sql, "steps_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients              : {cohort_patients:>9,}")
print(f"  ... with an index date     : {int(cov['patients_with_index_date']):>9,}")
print(f"  ... with >=1 step-day      : {int(cov['patients_with_any_steps']):>9,}"
      f"   ({int(cov['patients_with_any_steps']) / cohort_patients:6.1%} of cohort)")
print(f"step-days contributed        : {int(cov['step_days']):>9,}")

### How does coverage vary with distance from the surgery?

Wearable coverage is not flat in time, and the shape matters more than the headline: adherence
typically collapses in exactly the peri-operative days a recovery metric most needs. The
suppressed variant omits days with fewer than the threshold number of patients.

In [ ]:
curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM activity
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

steps_curve = cached_gbq(curve_sql, "steps_coverage_curve")
steps_curve["share_of_cohort"] = steps_curve["n_patients"] / cohort_patients

for label, (lo, hi) in WINDOWS.items():
    window = steps_curve[steps_curve["rel_day"].between(lo, hi)]
    if not window.empty:
        print(f"{label:9s} [{lo:>4}, {hi:>4}]  median patients/day = "
              f"{int(window['n_patients'].median()):>7,}"
              f"   ({window['share_of_cohort'].median():6.1%} of cohort)")

srp_common.plot_rel_day_curve(
    cfg, steps_curve, "n_patients",
    title="Patients with a Fitbit step record, by day relative to surgery",
    ylabel="patients with a step record", suppress=False, save_as="steps_coverage")
srp_common.plot_rel_day_curve(
    cfg, steps_curve, "n_patients",
    title="Patients with a Fitbit step record, by day relative to surgery",
    ylabel="patients with a step record", suppress=True, save_as="steps_coverage")

### How many patients have both a usable baseline and a usable follow-up?

A return-to-baseline outcome needs two windows populated for the same patient, so the binding
constraint is the paired count rather than either window alone. This is the single number that
decides whether a steps-based outcome is viable for this cohort.

In [ ]:
window_case = srp_common.window_case_sql("rel_day")

paired_sql = CTES + f"""
    , per_window AS (
        SELECT
            person_id,
            {window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM activity
        WHERE {window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    wide AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'acute',    n_days, 0)) AS acute_days,
            MAX(IF(window_name = 'early',    n_days, 0)) AS early_days,
            MAX(IF(window_name = 'recovery', n_days, 0)) AS recovery_days,
            MAX(IF(window_name = 'late',     n_days, 0)) AS late_days
        FROM per_window
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                                    AS patients_any_window,
        COUNTIF(baseline_days >= 7)                                 AS baseline_7d,
        COUNTIF(recovery_days >= 7)                                 AS recovery_7d,
        COUNTIF(baseline_days >= 7  AND recovery_days >= 7)         AS paired_7d,
        COUNTIF(baseline_days >= 14 AND recovery_days >= 14)        AS paired_14d,
        COUNTIF(baseline_days >= 7  AND late_days >= 7)             AS paired_late_7d
    FROM wide
"""

paired = cached_gbq(paired_sql, "steps_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "recovery_7d",
              "paired_7d", "paired_14d", "paired_late_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

print()
print("--> paired_7d is the analysable population for a return-to-baseline outcome.")

# 3-signal

### Does the step count actually move after surgery?

A metric that no surgery visibly perturbs cannot be a recovery outcome, however good its
coverage. Quantiles are computed inside BigQuery so only the summary crosses into the notebook.

In [ ]:
signal_sql = CTES + """
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                    AS n_patients,
        APPROX_QUANTILES(steps, 100)[OFFSET(25)]     AS p25_steps,
        APPROX_QUANTILES(steps, 100)[OFFSET(50)]     AS p50_steps,
        APPROX_QUANTILES(steps, 100)[OFFSET(75)]     AS p75_steps
    FROM activity
    WHERE rel_day BETWEEN -60 AND 180
    GROUP BY rel_day
    ORDER BY rel_day
"""

steps_signal = cached_gbq(signal_sql, "steps_signal_curve")

srp_common.plot_rel_day_curve(
    cfg, steps_signal, "p50_steps", band=("p25_steps", "p75_steps"),
    title="Median daily steps (IQR band), by day relative to surgery",
    ylabel="steps per day", suppress=False, save_as="steps_signal")
srp_common.plot_rel_day_curve(
    cfg, steps_signal, "p50_steps", band=("p25_steps", "p75_steps"),
    title="Median daily steps (IQR band), by day relative to surgery",
    ylabel="steps per day", suppress=True, save_as="steps_signal")

### How long does the median take to come back to baseline?

If the median never returns within the observed horizon, a "days to recovery" outcome is censored
for most patients and a fixed-horizon ratio is the better construction. The 7-day sustained rule
avoids calling a single good day a recovery.

In [ ]:
RECOVERY_FRACTION = 0.95
SUSTAIN_DAYS = 7

base_lo, base_hi = WINDOWS["baseline"]
baseline_level = steps_signal.loc[
    steps_signal["rel_day"].between(base_lo, base_hi), "p50_steps"].median()
target = RECOVERY_FRACTION * baseline_level

post = steps_signal[steps_signal["rel_day"] >= 0].reset_index(drop=True)
at_or_above = (post["p50_steps"] >= target).astype(int)
sustained = at_or_above.rolling(SUSTAIN_DAYS).sum() == SUSTAIN_DAYS

nadir_row = post.loc[post["p50_steps"].idxmin()]
print(f"baseline median steps (days {base_lo}..{base_hi}) : {baseline_level:,.0f}")
print(f"target ({RECOVERY_FRACTION:.0%} of baseline)                 : {target:,.0f}")
print(f"post-operative nadir                        : {nadir_row['p50_steps']:,.0f} "
      f"on day {int(nadir_row['rel_day'])}")

if sustained.any():
    # idxmax gives the row the window ENDS on; the run starts SUSTAIN_DAYS-1 rows earlier.
    # Indexing back by position rather than subtracting days keeps this correct when the
    # curve has gaps, which it will wherever a day fell below the suppression threshold.
    day = int(post.loc[sustained.idxmax() - SUSTAIN_DAYS + 1, "rel_day"])
    print(f"median first sustains the target on day     : {day}")
    days_to_recovery = day
else:
    print(f"median never sustains the target within day {int(post['rel_day'].max())} "
          "-- a time-to-recovery outcome would be censored for most patients")
    days_to_recovery = None

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="activity_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_steps"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["recovery"][1]),
    days_to_recovery=days_to_recovery,
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Non-wear is plausibly informative -- a patient recovering badly is less likely "
           "to wear the device -- so missingness must be treated as censoring, not noise."),
)